# Функциональное программирование

## Домашнее задание 3

**1.** Определите какие-нибудь (разумные) выражения, имеющие тип:
- a) `(b -> c, a -> b) -> a -> c` (буквы $a, b, c$ означают переменные по типам, по которым подразумеваются кванторы всеобщности);
- b) `[Double -> Double] -> Int -> Int`.

In [268]:
-- a) Композиция функций из пары
f1a :: (b -> c, a -> b) -> a -> c
f1a (f, g) x = f (g x)

-- Проверка типа
:t f1a

-- b) Длина списка функций + число
f1b :: [Double -> Double] -> Int -> Int
f1b fs n = length fs + n

-- Проверка типа
:t f1b

f1a :: forall b c a. (b -> c, a -> b) -> a -> c

f1b :: [Double -> Double] -> Int -> Int

**2.** Какой тип имеет терм `\f->f $ ($) $ (.)`? (Желательно указать наиболее общий тип.) Объясните, как прийти к такому выводу без помощи компилятора.

Определим тип терма:

1. Выражение под лямбдой разбирается c учетом порядка операций как
`f $ (($) $ (.))`, что редуцируется к `f (($) (.))`;

2. Тип `($)`:
`($) :: (a -> b) -> a -> b`. Значит `($) (.) = (.)`. Итого терм упрощается до
`\f -> f (.)`;

3. Тип `(.)`:
`(.) :: (b -> c) -> (a -> b) -> a -> c`;

4. Функция `f` принимает `(.)` в качестве аргумента, значит:
`f :: ((b -> c) -> (a -> b) -> a -> c) -> d`;

5. Итого, наиболее общий тип всего терма:
```
\f->f $ ($) $ (.) :: (((b -> c) -> (a -> b) -> a -> c) -> d) -> d
```

In [269]:
-- Компилятор говорит то же самое с точностью до обозначений переменных-типов:
:t \f -> f $ ($) $ (.)

\f -> f $ ($) $ (.) :: forall {b1} {c} {a} {b2}. (((b1 -> c) -> (a -> b1) -> a -> c) -> b2) -> b2

**3.** Определите следующие булевы функции с помощью механизма сравнения с образцом (pattern matching), не используя какие-либо иные, уже определенные, функции:
- а) исключающее или — используя не более трех образцов;
- b) функция большинства $maj_3$ (возвращает значение большинства своих аргументов) — используя не более четырех обрацов.

In [270]:
-- a) XOR (3 образца)
xor :: Bool -> Bool -> Bool
xor True True = False
xor False False = False
xor _ _ = True

-- Проверка
xor True True
xor True False
xor False True
xor False False

False

True

True

False

In [271]:
-- b) maj_3 (4 образца)
maj3 :: Bool -> Bool -> Bool -> Bool
maj3 True True _ = True
maj3 True _ True = True
maj3 _ True True = True
maj3 _ _ _ = False

-- Проверка
maj3 True True True
maj3 True True False
maj3 True False False
maj3 False False False

True

True

False

False

**4.** Определите функцию `f :: Integer -> Integer`, такую что $f(n) = 0^{fib(n)} + 1^{fib(n)} + \ldots + n^{fib(n)}$ при всех $n \geq 0$, где $fib(n)$ есть $n$-ое число Фибоначчии.

In [272]:
-- n-ое число Фибоначчи через хвостовую рекурсию
fib :: Integer -> Integer
fib n = fibImpl n 0 1
 where
  fibImpl 0 curr _ = curr
  fibImpl n curr next = fibImpl (n - 1) next (curr + next)

f4 :: Integer -> Integer
f4 n = sum [k ^ fib n | k <- [0..n]]

-- Проверка
f4 0
f4 1
f4 5

1

1

4425

**5.** Определите бесконечный список всех пифагоровых троек, т.е. троек вида ( $x, y, z$ ), где $x^2 + y^2 = z^2$, типа `(Integer, Integer, Integer)`.

In [273]:
pyTriples :: [(Integer, Integer, Integer)]
pyTriples = [(x, y, z) | z <- [1..], y <- [1..z], x <- [1..y], x^2 + y^2 == z^2]

take 10 pyTriples

[(3,4,5),(6,8,10),(5,12,13),(9,12,15),(8,15,17),(12,16,20),(15,20,25),(7,24,25),(10,24,26),(20,21,29)]

**6.** Натуральные числа $n$ и $m$ *дружественные*, если сумма собственных (т.е. меньших $n$ ) делителей числа $n$ равна $m$, и наоборот (например, 220 и 284 дружественные). Определите предикат, проверяющий пару натуральных чисел на дружественность.

In [274]:
-- Сумма собственных делителей числа n
divisorsSum :: Integer -> Integer
divisorsSum n = sum [d | d <- [1..n-1], n `mod` d == 0]

-- Предикат дружественности
friendly :: Integer -> Integer -> Bool
friendly n m = divisorsSum n == m && divisorsSum m == n

-- Проверка
friendly 220 284
friendly 1 2

True

False

**7.** Выясните, что делают библиотечные функции curry и uncurry, и реализуйте их.

In [275]:
-- curry преобразует функцию от пары в функцию двух аргументов
curry' :: ((a, b) -> c) -> a -> b -> c
curry' f x y = f (x, y)

-- uncurry преобразует функцию двух аргументов в функцию от пары
uncurry' :: (a -> b -> c) -> (a, b) -> c
uncurry' f (x, y) = f x y

-- Проверка
:t fst
:t curry' fst
curry' fst 1 2

:t (+)
:t uncurry' (+)
uncurry' (+) (3, 4)

fst :: forall a b. (a, b) -> a

curry' fst :: forall {c} {b}. c -> b -> c

1

(+) :: forall a. Num a => a -> a -> a

uncurry' (+) :: forall {c}. Num c => (c, c) -> c

7

**8.** Допустим, есть некоторые типы `A, B, C, D` и функции `g :: A -> B -> D` и `h :: D -> C`, а для функции `f :: A -> B -> C` выполнено тождество `f x y = h (g x y)`. Определите `f`, не упоминая локальных переменных (т.е. `х` и `у`), с помощью библиотечных функций `curry`, `uncurry` и `(.)`.

In [276]:
-- Вывод:
-- uncurry g :: (A, B) -> D
-- h . uncurry g :: (A, B) -> C
-- curry (h . uncurry g) :: A -> B -> C
-- Итого
f8 = curry (h8 . uncurry g8)

-- Проверка
:t (+)
:t (*2)
:t f8

f8 3 4 == (*2) ((+) 3 4)

(+) :: forall a. Num a => a -> a -> a

(*2) :: forall {a}. Num a => a -> a

f8 :: forall {c}. Num c => c -> c -> c

True

Типы А и В *изоморфны*, если существуют функции `f :: A -> B` и `g :: B -> A`, такие что верны равенства `f . g = id` и `g . f = id`.

**9.** Докажите, что для любых типов `А, В, С` изоморфны типы
- а) `C -> (A, B)` и `(C -> A, C -> B)`;
- b) `C -> (B -> A)` и `(B, C) -> A`.

In [277]:
-- a) Предъявим нужные f и g:

f9a :: (c -> (a, b)) -> (c -> a, c -> b)
f9a f = (\x -> fst (f x), \x -> snd (f x))

g9a :: (c -> a, c -> b) -> c -> (a, b)
g9a (g, h) x = (g x, h x)

-- Проверка:
test_f x = (x + 1, x * 2)

-- f . g = id
:t f9a . g9a
let pair = ((+1), (*2))
(fst ((f9a . g9a) pair)) 5 == 6
(snd ((f9a . g9a) pair)) 5 == 10

-- g . f = id
:t g9a . f9a
(g9a . f9a) test_f 5 == (6, 10)

f9a . g9a :: forall {c} {a} {b}. (c -> a, c -> b) -> (c -> a, c -> b)

True

True

g9a . f9a :: forall {c} {a} {b}. (c -> (a, b)) -> c -> (a, b)

True

In [278]:
-- b) Предъявим нужные f и g:

f9b :: (c -> b -> a) -> (b, c) -> a
f9b f (y, x) = f x y

g9b :: ((b, c) -> a) -> c -> b -> a
g9b f x y = f (y, x)

-- Проверка
-- f . g = id
:t f9b . g9b
(f9b . g9b) fst (3, 4) == 3

-- g . f = id
:t g9b . f9b
(g9b . f9b) (+) 3 4 == 7

f9b . g9b :: forall {b} {c} {a}. ((b, c) -> a) -> (b, c) -> a

True

g9b . f9b :: forall {c} {b} {a}. (c -> b -> a) -> c -> b -> a

True

10. С помощью `tуре` определите тип `Bfn` булевых функций трех аргументов.
- a) Определите на таких *функциях* структуру кольца, т.е. операции сложения и умножения *функций*, получающиеся поточечным применением исключающего или и конъюнкции соответственно, а также предикат равенства. Определите нуль, единицу и взятие противоположного элемента в таком кольце. Определите функцию, вычисляющую разумное представление элемента `Bfn` в виде строки.

- b) Поместив в начале модуля следующие директивы компилятора:

```
{-# LANGUAGE TypeSynonymInstances #-}
{-# LANGUAGE FlexibleInstances #-}
```

(позволяющие объявить экземпляром (instance) класса тип-синоним, вроде `Bfn`), с помощью механизма `instance` сделайте `Bfn` экземпляром классов `Eq`, `Num` и `Show` (в последнем случае нужно определить метод `show :: Bfn -> String`). Определения соответствующих методов должны быть возможно более разумными.

In [279]:
-- a)

type Bfn = Bool -> Bool -> Bool -> Bool

-- Сложение
addBfn :: Bfn -> Bfn -> Bfn
addBfn f g x y z = f x y z /= g x y z -- /= на Bool работает как XOR

-- Умножение
mulBfn :: Bfn -> Bfn -> Bfn
mulBfn f g x y z = f x y z && g x y z

-- Нуль кольца
zeroBfn :: Bfn
zeroBfn _ _ _ = False

-- Единица кольца
oneBfn :: Bfn
oneBfn _ _ _ = True

-- Взятие противоположного элемента
negBfn :: Bfn -> Bfn
negBfn f = f

-- Предикат равенства
eqBfn :: Bfn -> Bfn -> Bool
eqBfn f g = and [f a b c == g a b c | a <- [False, True], b <- [False, True], c <- [False, True]]

-- Строковое представление: таблица значений на всех 8 входах
showBfn :: Bfn -> String
showBfn f = [if f a b c then '1' else '0' | a <- [False, True], b <- [False, True], c <- [False, True]]

-- Проверка
showBfn zeroBfn
showBfn oneBfn
showBfn (\x y z -> x)
showBfn (addBfn oneBfn oneBfn)
eqBfn zeroBfn (addBfn oneBfn oneBfn)

"00000000"

"11111111"

"00001111"

"00000000"

True

In [280]:
-- b)

:extension TypeSynonymInstances
:extension FlexibleInstances

instance Eq Bfn where
 f == g = eqBfn f g

instance Num Bfn where
 f + g = addBfn f g
 f * g = mulBfn f g
 negate f = negBfn f
 abs f = f
 signum f = if f == zeroBfn then zeroBfn else oneBfn
 fromInteger n
    | even n = \_ _ _ -> False
    | otherwise = \_ _ _ -> True

instance Show Bfn where
 show f = showBfn f

-- Проверка
p1 = (\x y z -> x) :: Bfn
p2 = (\x y z -> y) :: Bfn

p1
p1 + p2
p1 * p2
p1 + p1
p1 + p1 == (0 :: Bfn)

00001111

00111100

00000011

00000000

True